# Module 3 — Tools for Dependency & License Analysis

This notebook demonstrates how to use **multiple tools together** to analyze ML dependencies and licenses, based on the demo script:

> "Tools for Dependency & License Analysis Demo — Demonstrates pip-audit, Safety, pip-licenses, and Snyk on real ML stacks"  
> "Recommendation: Use multiple tools together — pip-audit + Safety, pip-licenses, Snyk"

We’ll work with three sample ML stacks:

- `requirements_pytorch.txt` — PyTorch stack
- `requirements_tensorflow.txt` — TensorFlow stack
- `requirements_sklearn.txt` — scikit-learn stack

And we’ll walk through:

1. Loading the sample requirements files
2. Simulating `pip-audit` on the PyTorch stack
3. Simulating `Safety` on the TensorFlow stack
4. Simulating `pip-licenses` on the scikit-learn stack
5. Understanding Snyk’s role in ML supply chain security
6. Comparing the tools
7. CI/CD integration and best practices


## 1. Load sample ML requirements files

From the demo code:

> "[Creating Sample ML Requirements Files]"  
> "✓ Created requirements_pytorch.txt"  
> "✓ Created requirements_tensorflow.txt"  
> "✓ Created requirements_sklearn.txt"

We’ll load those files so we can inspect and analyze each stack.

In [ ]:
from pathlib import Path

base = Path('.')

def read_text(path: Path) -> str:
    return path.read_text(encoding='utf-8') if path.exists() else ''

req_pytorch = read_text(base / 'requirements_pytorch.txt')
req_tensorflow = read_text(base / 'requirements_tensorflow.txt')
req_sklearn = read_text(base / 'requirements_sklearn.txt')

print('Loaded requirements files:')
print('  requirements_pytorch.txt   :', bool(req_pytorch))
print('  requirements_tensorflow.txt:', bool(req_tensorflow))
print('  requirements_sklearn.txt   :', bool(req_sklearn))

### 1.1 Parse requirements into dependency lists

We’ll parse each requirements file into `(name, version)` pairs for easier analysis.

In [ ]:
import pandas as pd

def parse_requirements(text: str):
    deps = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        if '==' in line:
            name, version = line.split('==', 1)
            deps.append((name.strip(), version.strip()))
    return deps

deps_pytorch = parse_requirements(req_pytorch)
deps_tensorflow = parse_requirements(req_tensorflow)
deps_sklearn = parse_requirements(req_sklearn)

print('PyTorch stack:')
print(pd.DataFrame(deps_pytorch, columns=['package', 'version']))

print('\nTensorFlow stack:')
print(pd.DataFrame(deps_tensorflow, columns=['package', 'version']))

print('\nScikit-learn stack:')
print(pd.DataFrame(deps_sklearn, columns=['package', 'version']))

## 2. Tool 1 — pip-audit (PyTorch stack)

From the demo:

> "TOOL 1: PIP-AUDIT - Official Python Vulnerability Scanner"  
> "Features: Official tool from Python Packaging Authority, queries Python Packaging Advisory Database, fast and lightweight"  
> "[Scanning PyTorch Stack with pip-audit] Command: pip-audit -r requirements_pytorch.txt"

The script then shows an **expected output** when `pip-audit` is not installed, including vulnerabilities in `numpy`, `pillow`, and `torch`.

We’ll simulate that output as a DataFrame to make it easier to reason about in the notebook.

In [ ]:
pip_audit_findings = [
    {"name": "numpy", "version": "1.21.0", "id": "GHSA-cfnr-3xfx-3fmv", "fix_version": "1.21.2"},
    {"name": "pillow", "version": "8.2.0", "id": "CVE-2021-34552", "fix_version": "8.3.0"},
    {"name": "pillow", "version": "8.2.0", "id": "CVE-2021-23437", "fix_version": "8.2.1"},
    {"name": "torch", "version": "1.9.0", "id": "GHSA-47fc-vmwq-366v", "fix_version": "1.9.1"},
]

df_pip_audit = pd.DataFrame(pip_audit_findings)
df_pip_audit

### 2.1 Interpreting pip-audit results

From the demo explanation:

> "Each row shows: Package, Version, CVE ID, Fix Version"  
> "Action: Upgrade to Fix Version or later"

We can group by package to see how many issues each dependency introduces.

In [ ]:
df_pip_audit_summary = df_pip_audit.groupby('name').agg(
    vulns=('id', 'count'),
    min_fix_version=('fix_version', 'min')
).reset_index()
df_pip_audit_summary

## 3. Tool 2 — Safety (TensorFlow stack)

From the demo:

> "TOOL 2: SAFETY - Comprehensive Vulnerability Scanner"  
> "Features: Checks 51,000+ known vulnerabilities, curated vulnerability database, severity scoring (CVSS)"  
> "[Scanning TensorFlow Stack with Safety] Command: safety check -r requirements_tensorflow.txt"

The script then prints an **expected output** with vulnerabilities in `tensorflow`, `pillow`, and `urllib3`, including severity and fix versions.

We’ll simulate that as structured data.

In [ ]:
safety_findings = [
    {
        "package": "tensorflow",
        "version": "2.5.0",
        "vuln_id": "44715",
        "advisory": "TensorFlow vulnerable to heap buffer overflow",
        "severity": "HIGH",
        "fix_spec": "tensorflow>=2.5.1",
    },
    {
        "package": "pillow",
        "version": "8.1.0",
        "vuln_id": "44524",
        "advisory": "Pillow path traversal vulnerability",
        "severity": "HIGH",
        "fix_spec": "pillow>=8.2.0",
    },
    {
        "package": "urllib3",
        "version": "1.26.4",
        "vuln_id": "44525",
        "advisory": "urllib3 CRLF injection vulnerability",
        "severity": "MEDIUM",
        "fix_spec": "urllib3>=1.26.5",
    },
]

df_safety = pd.DataFrame(safety_findings)
df_safety

### 3.1 Safety vs pip-audit

From the demo:

> "pip-audit: Official Python vulnerability scanner"  
> "Safety: Checks 51,000+ known vulnerabilities, curated vulnerability database, severity scoring (CVSS)"  
> "More verbose output than pip-audit"

In practice:

- `pip-audit` is great for **fast, official** checks in CI
- `Safety` is great for **deeper coverage** and richer advisory details
- Using both gives better defense-in-depth for ML stacks


## 4. Tool 3 — pip-licenses (scikit-learn stack)

From the demo:

> "TOOL 3: PIP-LICENSES - License Compliance Analyzer"  
> "Identifies all package licenses, multiple output formats, critical for commercial ML products"  
> "[Scanning scikit-learn Stack for Licenses] Command: pip-licenses --from=mixed -p scikit-learn numpy scipy pandas"

The script then prints a **simulated license table** showing permissive licenses (BSD-3-Clause, PSF).

We’ll encode that as a DataFrame.

In [ ]:
pip_licenses_summary = [
    {"name": "scikit-learn", "version": "0.24.2", "license": "BSD-3-Clause"},
    {"name": "numpy", "version": "1.20.0", "license": "BSD-3-Clause"},
    {"name": "scipy", "version": "1.6.0", "license": "BSD-3-Clause"},
    {"name": "pandas", "version": "1.2.3", "license": "BSD-3-Clause"},
    {"name": "matplotlib", "version": "3.3.4", "license": "PSF"},
    {"name": "joblib", "version": "1.0.1", "license": "BSD-3-Clause"},
]

df_licenses = pd.DataFrame(pip_licenses_summary)
df_licenses

### 4.1 License risk assessment

From the demo:

> "Permissive Licenses (Low Risk): BSD-3-Clause, MIT, Apache-2.0, PSF"  
> "Restrictive Licenses (High Risk for Commercial): GPL-3.0, AGPL-3.0, LGPL"  
> "✓ All packages use permissive licenses — safe for commercial use"

We’ll tag each license as `permissive` or `restrictive` for quick policy checks.

In [ ]:
permissive = {"BSD-3-Clause", "MIT", "Apache-2.0", "PSF"}
restrictive = {"GPL-3.0", "AGPL-3.0", "LGPL"}

def classify_license(lic: str) -> str:
    if lic in permissive:
        return "permissive"
    if lic in restrictive:
        return "restrictive"
    return "unknown"

df_licenses["license_type"] = df_licenses["license"].apply(classify_license)
df_licenses

## 5. Tool 4 — Snyk (PyTorch stack)

From the demo:

> "TOOL 4: SNYK - Comprehensive Security Platform"  
> "Vulnerability + License scanning, dependency tree analysis, automated fix suggestions, continuous monitoring"  
> "Expected Snyk output: shows dependency path, severity, and remediation (upgrade torch, upgrade pillow)"

We’ll capture the key ideas:

- Snyk understands **dependency paths** (how a vulnerability is introduced)
- It suggests **coordinated upgrades** (e.g., upgrade `torch` to fix multiple issues)
- It integrates tightly with GitHub and CI/CD for **continuous monitoring**


### 5.1 Simulated Snyk findings (conceptual)

We’ll represent the Snyk example as a small table to show how vulnerabilities are traced through the dependency tree.

In [ ]:
snyk_findings = [
    {
        "package": "numpy",
        "severity": "MEDIUM",
        "description": "Buffer Overflow",
        "introduced_through": "torch@1.9.0 > numpy@1.21.0",
        "fix": "numpy@1.21.2",
    },
    {
        "package": "pillow",
        "severity": "HIGH",
        "description": "Path Traversal",
        "introduced_through": "torchvision@0.10.0 > pillow@8.2.0",
        "fix": "pillow@8.3.0",
    },
]

df_snyk = pd.DataFrame(snyk_findings)
df_snyk

## 6. Tool comparison

From the demo’s comparison table:

> "TOOL COMPARISON SUMMARY"  
> "Feature vs pip-audit, Safety, pip-lic, Snyk"  
> "Recommendation: Use multiple tools together — pip-audit: daily CI/CD, Safety: weekly deep scans, pip-licenses: before release, Snyk: continuous monitoring"

We’ll encode that comparison as a small DataFrame.

In [ ]:
tool_comparison = [
    {"feature": "Vulnerabilities", "pip_audit": "✓ Official", "safety": "✓ 51k+", "pip_licenses": "✗", "snyk": "✓ Best"},
    {"feature": "Licenses", "pip_audit": "✗", "safety": "✗", "pip_licenses": "✓ Only", "snyk": "✓"},
    {"feature": "Dep Tree", "pip_audit": "✗", "safety": "✗", "pip_licenses": "✗", "snyk": "✓"},
    {"feature": "Auto Fix", "pip_audit": "✗", "safety": "✗", "pip_licenses": "✗", "snyk": "✓"},
    {"feature": "CI/CD", "pip_audit": "✓ Easy", "safety": "✓ Easy", "pip_licenses": "✓ Easy", "snyk": "✓ Best"},
    {"feature": "Free", "pip_audit": "✓ Always", "safety": "✓ OSS", "pip_licenses": "✓ Always", "snyk": "✓ OSS"},
    {"feature": "Speed", "pip_audit": "Fast", "safety": "Fast", "pip_licenses": "Fast", "snyk": "Slower"},
    {"feature": "Account", "pip_audit": "✗ No", "safety": "✗ No", "pip_licenses": "✗ No", "snyk": "✓ Yes"},
]

df_tools = pd.DataFrame(tool_comparison)
df_tools

## 7. CI/CD integration

        "CI/CD INTEGRATION"  
> "GitHub Actions workflow: runs on every PR and push, daily scheduled scans, uses all 4 tools, fails build on critical issues"

The demo provides a full GitHub Actions workflow that:

- Installs `pip-audit`, `safety`, `pip-licenses`
- Runs them against `requirements.txt`
- Invokes Snyk via a GitHub Action

We’ll embed that workflow here as a reference string.

In [ ]:
github_workflow = '''
# .github/workflows/dependency-scan.yml
name: Dependency Security Scan

on:
  pull_request:
  push:
    branches: [main]
  schedule:
    - cron: '0 0 * * *'  # Daily

jobs:
  scan:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      
      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: '3.9'
      
      - name: Install scanning tools
        run: |
          pip install pip-audit safety pip-licenses
      
      - name: Run pip-audit
        run: pip-audit -r requirements.txt
        
      - name: Run Safety
        run: safety check -r requirements.txt --json
        
      - name: Check licenses
        run: pip-licenses --fail-on="GPL;AGPL"
        
      - name: Snyk scan
        uses: snyk/actions/python@master
        env:
          SNYK_TOKEN: ${{ secrets.SNYK_TOKEN }}
'''

print(github_workflow)

### 7.1 Tool-specific configuration

From the demo:

> "pip-audit can be configured to fail only on HIGH+ severity"  
> "Safety allows ignoring specific vulnerabilities with justification"  
> "Snyk has project-level policies for organization-wide rules"

We’ll capture the example configs as strings for reference.

In [ ]:
pipaudit_config = '''
# .pip-audit.yml
fail-on: HIGH
ignore-vulns:
  - GHSA-xxxx-xxxx-xxxx  # False positive for our use case
'''

safety_config = '''
# .safety-policy.yml
security:
  ignore-vulnerabilities:
    - 12345  # Accepted risk - documented in security review
  continue-on-vulnerability-error: false
'''

print('pip-audit config example:')
print(pipaudit_config)

print('\nSafety config example:')
print(safety_config)

## 8. Best practices for ML dependency & license analysis

From the demo’s **BEST PRACTICES** section:

> "1. Use Multiple Tools (Defense in Depth) — No single tool catches everything"  
> "2. Automate in CI/CD — Run on every pull request, daily scheduled scans"  
> "3. Prioritize Fixes — CRITICAL: < 24 hours, HIGH: 1 week, MEDIUM: 1 month"  
> "4. Continuous Monitoring — Dependabot/Renovate, alerts, monthly reviews"  
> "5. Documentation — Document accepted risks, license decisions, update logs"

For AI/ML systems, this translates to:

- **Always** scan your ML stacks (PyTorch, TensorFlow, scikit-learn) with at least two vulnerability tools
- **Always** run license checks before releases, especially for commercial deployments
- **Always** integrate these tools into CI/CD, not just local dev
- **Always** document exceptions and accepted risks


## 9. Summary — Why multiple tools matter

This module’s core message is simple:

- `pip-audit` gives you **official, fast** vulnerability checks
- `Safety` gives you **broader coverage and richer advisories**
- `pip-licenses` ensures **license compliance** for ML stacks
- `Snyk` adds **dependency tree analysis, remediation guidance, and continuous monitoring**

Relying on a single tool is like running an antivirus with half the signatures missing.

For ML supply chain security, you need **defense in depth**:

1. Multiple scanners (pip-audit + Safety + Snyk)
2. License analysis (pip-licenses)
3. CI/CD enforcement
4. Continuous monitoring and documented risk decisions
